In [ ]:

import sys
sys.path.append("..")

In [ ]:

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch
from tqdm import tqdm
from  ddpm_backtest.data_loaders import get_nifty_regime_data,compute_regime_score,fit_garch_volatility,prepare_dataloaders
from ddpm_backtest.noising_time import set_seed,cosine_beta_scheduler,timestep_embedding,noisify,split_context
from ddpm_backtest.models import FiLMLayer,f_net
from ddpm_backtest.diffusion_utils import TabularDDPM,EMA


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape: (2566, 35)

Regime counts:
regime
0    1787
1     779
Name: count, dtype: int64
MARKET_DIM=25, TIME_DIM=8, CONTEXT_DIM=33
Device: cpu | T=100


In [4]:
risk_df = get_nifty_regime_data()
risk_df['regime_score'] = compute_regime_score(risk_df)
risk_df['regime_score'] = risk_df['regime_score'].fillna(0.5)
risk_df['regime'] = (risk_df['regime_score'] > 0.5).astype(int)
risk_df.drop(columns=['sma_200','Close'], inplace=True)
risk_df = fit_garch_volatility(risk_df)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Fitting GARCH(1,1) on training data...
                     Constant Mean - GARCH Model Results                      
Dep. Variable:          target_return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2619.48
Distribution:                  Normal   AIC:                           5246.95
Method:            Maximum Likelihood   BIC:                           5269.46
                                        No. Observations:                 2052
Date:                Sun, Apr 26 2026   Df Residuals:                     2051
Time:                        10:43:23   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0776  1.

In [5]:
risk_df.head()

,target_return,return_lag_1,return_lag_2,return_lag_3,return_lag_5,return_lag_10,return_lag_21,cum_return_5d,cum_return_10d,cum_return_21d,...,VIX,RSI,skew,kurtosis,vol_spike,RSI_velocity,regime,regime_score,garch_vol,garch_resid
Date,,,,,,,,,,,,,,,,,,,,,
2015-10-27,-0.007523,-0.004216,0.005288,-0.001205,0.004469,-0.005645,0.004336,0.002715,0.008614,0.055830,...,17.230000,63.648657,-1.867438,7.837366,0.923993,-16.027928,0,0.500000,0.006987,-1.076581
2015-10-28,-0.007302,-0.003353,-0.004216,0.005288,-0.001621,-0.001462,0.002870,-0.005107,0.010906,0.048141,...,16.540001,57.820192,-1.840251,7.784645,1.129109,-17.032109,0,0.467612,0.007908,-0.923363
2015-10-29,-0.005681,-0.007523,-0.003353,-0.004216,-0.001205,-0.002931,-0.009295,-0.011009,0.004846,0.037748,...,17.040001,49.435055,-1.787560,7.617661,1.279939,-27.030146,0,0.476838,0.007990,-0.711001
2015-10-30,-0.001861,-0.007302,-0.007523,-0.003353,0.005288,0.008792,0.006087,-0.017106,0.000475,0.039741,...,17.580000,48.428844,-1.770332,7.721606,1.182082,-15.219813,0,0.489299,0.008040,-0.231521
2015-11-02,0.001229,-0.005681,-0.007302,-0.007523,-0.004216,0.007145,0.013374,-0.028074,-0.013998,0.027973,...,17.879999,38.647577,-1.738055,7.611434,0.809108,-19.172615,0,0.497006,0.007944,0.154702


In [6]:
PAST_RETURN_COLS   = ([f"return_lag_{l}" for l in [1, 2, 3, 5, 10, 21]] +
                      [f"cum_return_{w}d" for w in [5, 10, 21, 63]])
RVOL_COLS          = ([f"rvol_{w}d" for w in [5, 10, 21, 63]] +
                      ["vol_ratio_5_21", "vol_ratio_21_63"])
ORIGINAL_CONT_COLS = ["realized_vol", "VIX", "RSI", "skew",
                       "kurtosis", "vol_spike", "RSI_velocity", "garch_vol"]
UNSCALED_CONT_COLS = ["regime_score",
                       "dow_sin", "dow_cos", "month_sin", "month_cos",
                       "dom_sin", "dom_cos", "quarter_sin", "quarter_cos"]
SCALED_CONT_COLS   = ORIGINAL_CONT_COLS + PAST_RETURN_COLS + RVOL_COLS

MARKET_DIM  = len(SCALED_CONT_COLS) + 1
TIME_DIM    = len(UNSCALED_CONT_COLS) - 1
CONTEXT_DIM = len(SCALED_CONT_COLS) + len(UNSCALED_CONT_COLS)
print(f"MARKET_DIM={MARKET_DIM}, TIME_DIM={TIME_DIM}, CONTEXT_DIM={CONTEXT_DIM}")

MARKET_DIM=25, TIME_DIM=8, CONTEXT_DIM=33


In [7]:
train_dl, val_dl, condition_df, scaler_x, scaler_cond, val_df = prepare_dataloaders(risk_df)
print(f"\nscaler_x mean={scaler_x.mean_[0]:.4f}  scale={scaler_x.scale_[0]:.4f}  (mean~0, scale~1)")

Train: 2,052  Val: 257  Test: 257

scaler_x mean=0.0591  scale=1.0063  (mean~0, scale~1)


In [8]:
T          = 100
betas      = cosine_beta_scheduler(T)
alphas     = 1 - betas
alphas_bar = torch.cumprod(torch.from_numpy(alphas).float(), dim=0)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
alphas_bar = alphas_bar.to(device)
print(f"Device: {device} | T={T}")

Device: cpu | T=100


In [9]:
diffusion_model = TabularDDPM(
    d_in=1, cond_in_classes=2,
    scaled_cont_dim=len(SCALED_CONT_COLS),
    fixed_market_dim=1, time_dim=TIME_DIM,
    t_dim=64, dropout=0.1,
).to(device)
print(f"Parameters: {sum(p.numel() for p in diffusion_model.parameters()):,}")

optimizer        = torch.optim.Adam(diffusion_model.parameters(), lr=3e-4, weight_decay=1e-4)
num_epochs       = 500
p_uncond         = 0.15
warmup_eps       = 10
patience         = 25
best_val_mean    = np.inf
patience_counter = 0
ema_shadow_best  = None

def lr_lambda(epoch):
    if epoch < warmup_eps:
        return (epoch + 1) / warmup_eps
    progress = (epoch - warmup_eps) / max(1, num_epochs - warmup_eps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
ema       = EMA(diffusion_model)

set_seed(42)
for epoch in range(num_epochs):
    diffusion_model.train()
    train_loss, n_train = 0.0, 0
    for x0, c, ctx in tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
        x0, c, ctx = x0.to(device), c.to(device), ctx.to(device)
        if x0.dim() == 1: x0 = x0.unsqueeze(1)
        mc, tf     = split_context(ctx)
        drop_mask  = torch.rand(c.shape[0], device=device) < p_uncond
        xt, t, eps = noisify(T, x0, alphas_bar)
        eps_pred   = diffusion_model(xt, c, mc, tf, t.float(), drop_mask)
        snr     = alphas_bar[t] / (1.0 - alphas_bar[t])
        weights = (snr / snr.mean()).clamp(0.1, 10.0).unsqueeze(1)
        loss    = (weights * F.smooth_l1_loss(eps_pred, eps, reduction="none")).mean()
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
        optimizer.step(); ema.update(diffusion_model)
        train_loss += loss.item(); n_train += 1

    diffusion_model.eval()
    val_loss, n_val = 0.0, 0
    with torch.no_grad():
        for x0v, cv, ctxv in val_dl:
            x0v, cv, ctxv = x0v.to(device), cv.to(device), ctxv.to(device)
            if x0v.dim() == 1: x0v = x0v.unsqueeze(1)
            mcv, tfv = split_context(ctxv)
            xtv, tv, epsv = noisify(T, x0v, alphas_bar)
            val_loss += F.smooth_l1_loss(
                diffusion_model(xtv, cv, mcv, tfv, tv.float()), epsv).item()
            n_val += 1

    val_mean = val_loss / n_val
    if val_mean < best_val_mean:
        best_val_mean    = val_mean
        patience_counter = 0
        torch.save(diffusion_model.state_dict(), "best_model.pt")
        ema_shadow_best  = {k: v.clone() for k, v in ema.shadow.items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d} | Train {train_loss/n_train:.6f} | "
              f"Val {val_mean:.6f} | LR {scheduler.get_last_lr()[0]:.2e} | "
              f"Patience {patience_counter}/{patience}")

diffusion_model.load_state_dict(torch.load("best_model.pt"))
ema.shadow = ema_shadow_best
print(f"\nLoaded best model (val={best_val_mean:.6f})")

Parameters: 117,441


Epoch 010 | Train 0.244615 | Val 0.399610 | LR 3.00e-04 | Patience 8/25


Epoch 020 | Train 0.315394 | Val 0.337597 | LR 3.00e-04 | Patience 2/25


Epoch 030 | Train 0.266187 | Val 0.438128 | LR 2.99e-04 | Patience 1/25


Epoch 040 | Train 0.326669 | Val 0.573078 | LR 2.97e-04 | Patience 2/25


...

Epoch 110 | Train 0.258881 | Val 0.176554 | LR 2.70e-04 | Patience 13/25


Epoch 120 | Train 0.253637 | Val 0.184822 | LR 2.64e-04 | Patience 2/25


Epoch 130 | Train 0.294271 | Val 0.138854 | LR 2.58e-04 | Patience 12/25


Epoch 140 | Train 0.229442 | Val 0.511753 | LR 2.51e-04 | Patience 22/25


Early stopping at epoch 143

Loaded best model (val=0.135164)


In [10]:
def check_noise_prediction(diffusion_model,train_dl,alphas_bar,device):    
    diffusion_model.eval()
    with torch.no_grad():
        for x0, c, ctx in train_dl:
            x0, c, ctx = x0.to(device), c.to(device), ctx.to(device)
            if x0.dim() == 1: x0 = x0.unsqueeze(1)
            mc, tf = split_context(ctx)
            print(f"{'Timestep':<12} {'Loss':>10}")
            print("-" * 24)
            for t_check in [5, 25, 50, 75, 95]:
                ab    = alphas_bar[t_check]
                eps   = torch.randn_like(x0)
                xt    = torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * eps
                t_vec = torch.full((x0.shape[0],), t_check, device=device, dtype=torch.float)
                pred  = diffusion_model(xt, c, mc, tf, t_vec)
                print(f"  t={t_check:<8} {F.smooth_l1_loss(pred, eps).item():>10.6f}")
            print("\nExpected: loss highest at t=5, lowest toward t=95")
            break
check_noise_prediction(diffusion_model, train_dl, alphas_bar, device)

Timestep           Loss
------------------------
  t=5          0.378357
  t=25         0.340462
  t=50         0.192290
  t=75         0.071854
  t=95         0.009356

Expected: loss highest at t=5, lowest toward t=95
